In [2]:
from feature_dynamics import Config, PromptGenerator, DataCollector
from feature_dynamics import TokenOnlyPredictor, StateTokenPredictor
from feature_dynamics import evaluate_predictors


In [4]:
dc = DataCollector(Config())
model = dc.model
lm = model.model.language_model


Loading model google/gemma-3-27b-it...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/12 [00:00<?, ?it/s]

Loading SAE for layer 31...


In [19]:
# This should now work with our updated code logic                                                                                                                                             
if hasattr(model.config, 'text_config') and hasattr(model.config.text_config, 'layer_types'):                                                                                                  
    layer_types = model.config.text_config.layer_types                                                                                                                                         
    print(f"Found {len(layer_types)} layer types")                                                                                                                                             
    print(f"Full attention layers: {[i for i, t in enumerate(layer_types) if t == 'full_attention']}") 

Found 62 layer types
Full attention layers: [5, 11, 17, 23, 29, 35, 41, 47, 53, 59]


In [17]:
# Check what attributes exist                                                                                                                                                                  
dir(model.config)                                                                                                                                                                              
                                                                                                                                                                                                
# Check specifically for layer-related attributes                                                                                                                                              
print(f"num_hidden_layers: {model.config.num_hidden_layers}")                                                                                                                                  
                                                                                                                                                                                                
# Check if there's a sliding window attribute                                                                                                                                                  
print(f"sliding_window: {getattr(model.config, 'sliding_window', 'NOT FOUND')}")                                                                                                               
                                                                                                                                                                                                
# Check the actual number of layers in the model                                                                                                                                               
print(f"Number of layers: {len(model.model.language_model.layers)}")                                                                                                                           
                                                                                                                                                                                                
# Check if layers have attention_type attribute                                                                                                                                                
print(f"Layer 0 has attention_type: {hasattr(model.model.language_model.layers[0], 'attention_type')}")                                                                                        
if hasattr(model.model.language_model.layers[0], 'attention_type'):                                                                                                                            
    print(f"Layer 0 attention_type: {model.model.language_model.layers[0].attention_type}") 

AttributeError: 'Gemma3Config' object has no attribute 'num_hidden_layers'

In [13]:
layers = lm.layers
layers[5]

Gemma3DecoderLayer(
  (self_attn): Gemma3Attention(
    (q_proj): Linear(in_features=5376, out_features=4096, bias=False)
    (k_proj): Linear(in_features=5376, out_features=2048, bias=False)
    (v_proj): Linear(in_features=5376, out_features=2048, bias=False)
    (o_proj): Linear(in_features=4096, out_features=5376, bias=False)
    (q_norm): Gemma3RMSNorm((128,), eps=1e-06)
    (k_norm): Gemma3RMSNorm((128,), eps=1e-06)
  )
  (mlp): Gemma3MLP(
    (gate_proj): Linear(in_features=5376, out_features=21504, bias=False)
    (up_proj): Linear(in_features=5376, out_features=21504, bias=False)
    (down_proj): Linear(in_features=21504, out_features=5376, bias=False)
    (act_fn): GELUTanh()
  )
  (input_layernorm): Gemma3RMSNorm((5376,), eps=1e-06)
  (post_attention_layernorm): Gemma3RMSNorm((5376,), eps=1e-06)
  (pre_feedforward_layernorm): Gemma3RMSNorm((5376,), eps=1e-06)
  (post_feedforward_layernorm): Gemma3RMSNorm((5376,), eps=1e-06)
)

In [9]:
layer = layers[0]
help(layer.forward)

Help on method forward in module transformers.models.gemma3.modeling_gemma3:

forward(hidden_states: torch.Tensor, position_embeddings_global: torch.Tensor, position_embeddings_local: torch.Tensor, attention_mask: Optional[torch.Tensor] = None, position_ids: Optional[torch.LongTensor] = None, past_key_values: Optional[transformers.cache_utils.Cache] = None, output_attentions: Optional[bool] = False, use_cache: Optional[bool] = False, cache_position: Optional[torch.LongTensor] = None, **kwargs) -> tuple[torch.FloatTensor, typing.Optional[tuple[torch.FloatTensor, torch.FloatTensor]]] method of transformers.models.gemma3.modeling_gemma3.Gemma3DecoderLayer instance
    Define the computation performed at every call.

    Should be overridden by all subclasses.

    .. note::
        Although the recipe for forward pass needs to be defined within
        this function, one should call the :class:`Module` instance afterwards
        instead of this since the former takes care of running the


In [20]:
from sae_lens import SAE                                                                                                                                                                       
                                                                                                                                                                                                
# Load a small SAE to inspect                                                                                                                                                                  
sae, cfg_dict, sparsity = SAE.from_pretrained(                                                                                                                                                 
    release="gemma-scope-2-27b-it-res",                                                                                                                                                        
    sae_id="layer_31_width_16k_l0_medium",                                                                                                                                                     
    device="cpu"                                                                                                                                                                               
)                                                                                                                                                                                              
                                                                                                                                                                                                
# Check attributes                                                                                                                                                                             
print("SAE attributes:")                                                                                                                                                                       
print([attr for attr in dir(sae) if not attr.startswith('_')])                                                                                                                                 
                                                                                                                                                                                                
# Check weight shapes                                                                                                                                                                          
print(f"\nW_enc shape: {sae.W_enc.shape}")                                                                                                                                                     
print(f"b_enc shape: {sae.b_enc.shape}")                                                                                                                                                       
                                                                                                                                                                                                
# Check if decoder exists                                                                                                                                                                      
print(f"Has W_dec: {hasattr(sae, 'W_dec')}")                                                                                                                                                   
print(f"Has b_dec: {hasattr(sae, 'b_dec')}")                                                                                                                                                   
                                                                                                                                                                                                
# Check for decode method                                                                                                                                                                      
print(f"Has decode method: {hasattr(sae, 'decode')}")                                                                                                                                          
                                                        

SAE attributes:
['T_destination', 'W_dec', 'W_enc', 'activation_fn', 'add_caching_hooks', 'add_hook', 'add_module', 'add_perma_hook', 'apply', 'b_dec', 'b_enc', 'bfloat16', 'buffers', 'cache_all', 'cache_some', 'call_super_init', 'cfg', 'check_and_add_hook', 'check_hooks_to_add', 'children', 'clear_contexts', 'compile', 'context_level', 'cpu', 'cuda', 'd_head', 'decode', 'device', 'double', 'dtype', 'dump_patches', 'encode', 'eval', 'extra_repr', 'float', 'fold_W_dec_norm', 'fold_activation_norm_scaling_factor', 'forward', 'from_dict', 'from_pretrained', 'from_pretrained_with_cfg_and_sparsity', 'get_activation_fn', 'get_buffer', 'get_caching_hooks', 'get_extra_state', 'get_name', 'get_parameter', 'get_sae_class_for_architecture', 'get_sae_config_class_for_architecture', 'get_submodule', 'half', 'hook_dict', 'hook_points', 'hook_sae_acts_post', 'hook_sae_acts_pre', 'hook_sae_error', 'hook_sae_input', 'hook_sae_output', 'hook_sae_recons', 'hook_z_reshaping_mode', 'hooks', 'initialize_wei

/tmp/ipykernel_1188679/2797730185.py:4: DeprecationWarning: Unpacking SAE objects is deprecated. SAE.from_pretrained() now returns only the SAE object. Use SAE.from_pretrained_with_cfg_and_sparsity() to get the config dict and sparsity as well.
  sae, cfg_dict, sparsity = SAE.from_pretrained(


In [ ]:
print(f"W_dec shape: {sae.W_dec.shape}")                                                                                                                                                       
print(f"b_dec shape: {sae.b_dec.shape}") 

W_dec shape: torch.Size([16384, 5376])
b_dec shape: torch.Size([5376])


: 